# Day 4 — ILT 1: Ingestion Patterns Recap — Across GlobalMart's Sources
### GlobalMart Data Engineering · 12:00 PM – 1:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- State GlobalMart's two real ingestion pathways and which tables travel through each
- Explain why Postgres (Supabase) uses CDC via Lakeflow Connect, not a nightly full read
- Explain why the four ADLS file-drop entities use Auto Loader, not a plain `spark.read` loop
- Explain why the REST API and Neo4j graph patterns from Day 3 are **not** part of this production pipeline
- Describe how both pathways land in Bronze with the same audit-column contract
- Trace the path from raw source all the way to `fact_sales`

---

## Agenda

| Time | Topic |
|------|-------|
| 12:00 | GlobalMart architecture recap — 2 pathways, 6 tables |
| 12:10 | Pathway 1 — Postgres (Supabase) via Lakeflow Connect CDC |
| 12:25 | Pathway 2 — ADLS file drops via Auto Loader |
| 12:40 | Why not APIs / Neo4j? (Day 3 side-explorations, not in this pipeline) |
| 12:47 | Cross-pathway comparison + Bronze landing map |
| 12:55 | Q&A |

> **Note on "4 sources":** you'll sometimes hear this session called "all 4 sources." That name is a holdover from an earlier design that had 4 separate *systems* (Postgres, a REST API, Neo4j, ADLS). The confirmed architecture has only **two real systems** — Postgres and ADLS. The "4" that survives is accurate at the *file* level: 4 flat files drop into ADLS (`products`, `customers`, `address`, `payments`). Postgres contributes 2 more tables (`orders`, `order_items`) via CDC. Six tables, two pathways — that's what we're recapping today.

---
## GlobalMart Architecture — Quick Recap

```
┌───────────────────────────────────────────────────────────────────────┐
│                       GLOBALMART'S TWO REAL SOURCES                     │
│                                                                         │
│   ┌─────────────────────────┐          ┌───────────────────────────┐  │
│   │  Postgres (Supabase)    │          │  ADLS File Drops           │  │
│   │  orders, order_items    │          │  products, customers,      │  │
│   │                         │          │  address, payments         │  │
│   └────────────┬────────────┘          └──────────────┬──────────────┘  │
│                │  Lakeflow Connect                     │  Auto Loader   │
│                │  (query/cursor-based)                 │  (cloudFiles)  │
│                ▼                                       ▼               │
└────────────────┼───────────────────────────────────────┼───────────────┘
                  │                                       │
                  ▼                                       ▼
┌───────────────────────────────────────────────────────────────────────┐
│                       BRONZE LAYER (Delta Lake)                        │
│   bronze/supabase/orders        bronze/adls/products                   │
│   bronze/supabase/order_items   bronze/adls/customers                  │
│                                  bronze/adls/address                    │
│                                  bronze/adls/payments                   │
└───────────────────────────────────────────────────────────────────────┘
                  │
                  ▼
           SILVER → GOLD (star schema) → fact_sales → Genie
```

### Why Only Two Real Pathways?

GlobalMart's actual production footprint is simple on purpose: an OLTP database for transactional data, and flat files for everything suppliers/ops teams hand off in bulk.

| Pathway | Tables | Why This Pathway |
|---------|--------|-------------------|
| **Postgres CDC** (Lakeflow Connect) | `orders`, `order_items` | High-change-rate transactional rows — need inserts and updates captured continuously (this pipeline is configured query/cursor-based, so hard deletes are the one gap — see Day 2) |
| **ADLS Autoloader** | `products`, `customers`, `address`, `payments` | Reference/dimension-shaped data that arrives as periodic file drops — need exactly-once file processing with schema evolution |

You met both mechanisms already: Day 2 (`Day2_1_ILT1_CDC_Concepts_WAL_Supabase`, `Day2_2_ILT2_Lakeflow_Connect_Storage_Credentials`, `Day2_4_HOL2_CDC_PostgreSQL_WAL`) walked the general WAL/CDC mental model by hand and then GlobalMart's real, query/cursor-based Lakeflow Connect pipeline, and Day 3 (`Day3_3_ILT2_Schema_Evolution_Concepts`, `Day3_4_DEMO_AutoLoader_v2`) covered Auto Loader and schema evolution concepts with a live demo — but on disposable demo/sandbox data, not GlobalMart's real Bronze tables. Today's Bronze HOL (right after this session) is where `products`, `customers`, `address`, and `payments` actually get built as real Bronze tables for the first time. This ILT's job is to make sure the *shape* of both pathways — CDC and Autoloader — is crystal clear before that hands-on build.

---
## Pathway 1 — Postgres (Supabase) via Lakeflow Connect CDC

### What Is It?
Supabase is GlobalMart's hosted PostgreSQL database — the transactional core. Every order and every order line item is written here first.

### The Ingestion Challenge
```
Problem: Postgres is an OLTP database — optimised for fast writes, not bulk reads.
         Querying the full orders table (millions of rows) every hour kills performance.
         We need CHANGES ONLY — what was inserted/updated since last run.
```

### The Mechanism — Query/Cursor-Based Capture

```
Supabase PostgreSQL
      |
      ▼  Lakeflow Connect queries WHERE updated_at > last_seen_value on each run
Lakeflow Connect (managed, production) — orders_data_ingestion_cdc pipeline
      |
      ▼  INSERT / UPDATE events upserted (history tracking: Off / SCD1)
gbmart.bronze.orders , gbmart.bronze.order_items
      |
      ▼  MERGE (upsert) in Silver
silver.orders_clean , silver.order_items_clean
```

**Key concepts (recap from Day 2):**
- **Cursor column** = the column Lakeflow Connect re-queries against each run to find what changed — GlobalMart's real pipeline uses `updated_at`
- **Lakeflow Connect** is the managed pipeline GlobalMart uses in production — it re-queries the source table on the cursor column and lands upserted rows straight into Bronze Delta tables. No JDBC code, no manual query-scheduling
- Lakeflow Connect *can* also run log-based (reading the WAL directly, which does catch hard deletes) — but GlobalMart's real pipeline is configured query/cursor-based, so it **cannot detect hard deletes**: a deleted source row simply stops matching the query and goes stale in Bronze
- Each row lands upserted (SCD1, latest state only) — no `_cdc_op` column, since query-based capture only ever sees current rows, not a stream of discrete change events

**Trigger mode:** GlobalMart's real pipeline has no schedule — triggered manually (see Day 2 ILT 2).

### Why Not Just a Nightly JDBC Pull?

| | Lakeflow Connect (query/cursor) | Plain JDBC + watermark |
|--|-----------|------------|
| Captures deletes? | No — same blind spot as any cursor-based approach | No — a deleted row leaves no trace |
| Source load | Low (managed, scheduled queries) | Medium (hand-rolled script) |
| Latency | Depends on run trigger | Depends on schedule |
| What GlobalMart uses in production | ✅ Preferred (managed) | Fallback only — what this is automating away |

> Day 2 walked both the general WAL/CDC mental model (log-based, by hand via JDBC) and GlobalMart's actual query/cursor-based Lakeflow Connect pipeline in detail — today's recap assumes that context.

---
## Pathway 2 — ADLS File Drops via Auto Loader

### What Is It?
Four flat files land in the `raw-data/` folder of GlobalMart's shared ADLS Gen2 external location: `products.csv`, `customers.csv`, `addresses.csv`, `payments.csv`. These aren't transactional events — they're periodic drops of reference/dimension-shaped data (product catalog updates, customer master data, address book, payment records).

> **Real path shape:** `abfss://<your-container>@<your-storage-account>.dfs.core.windows.net/raw-data/<entity>/` — registered as a Unity Catalog external location. Landing tables are Unity Catalog managed tables: `<your-catalog>.bronze.customers`, `<your-catalog>.bronze.products`, `<your-catalog>.bronze.addresses`, `<your-catalog>.bronze.payments`.

### The Ingestion Challenge
```
Problem: Files land at unpredictable times.
         Must process each file EXACTLY ONCE — no re-reads, no duplicate rows.
         Schema can evolve (a supplier adds a new column to products.csv).
         Volume varies wildly file to file.
```

### Pattern — Auto Loader (`cloudFiles`)

```python
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferColumnTypes", "true")
    .load(f"{EXTERNAL_LOCATION}/products/")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.bronze.products")
)
```

### What Makes Auto Loader Special (recap from Day 3)

```
Without Auto Loader (spark.read loop):
  Run 1: reads products.csv → 12,000 rows
  Run 2: reads products.csv AGAIN → 12,000 DUPLICATE rows
  Problem: every run re-processes every file

With Auto Loader (cloudFiles):
  Run 1: reads products.csv → 12,000 rows → checkpoint records products.csv
  Run 2: sees products.csv in checkpoint → SKIP
          sees a new file → processes only the new rows
  Result: zero duplicates, exactly-once delivery
```

**Trigger modes:**
- `availableNow=True` → process all pending files, then stop (scheduled batch style) — what we use for GlobalMart's periodic drops
- `processingTime="30 seconds"` → continuous, detect files as they land

> Day 3 covered these Auto Loader mechanics conceptually and via a live demo on disposable sandbox data (`Day3_3_ILT2_Schema_Evolution_Concepts`, `Day3_4_DEMO_AutoLoader_v2`). Today's Bronze HOL is where this exact pattern gets built for real, against all four ADLS entities: `products`, `customers`, `addresses`, `payments`.

---
## Why Not APIs or Neo4j? (Day 3 Side-Explorations)

If you've heard GlobalMart described elsewhere as having "4 source *systems*" — Postgres, a REST API, Neo4j, and ADLS — that was the earlier design. The confirmed architecture (`globalmart_problem_statement_architecture.html`) is explicit:

> *"Two real sources, the way GlobalMart actually has them today. A REST API and a graph database are explored separately on Day 3 as patterns you'll meet in other projects — they don't feed this pipeline."*

### What Day 3 ILT 1 Covered (and why it's still valuable)

| Pattern | What You Learned | Why It's Not in *This* Pipeline |
|---------|-------------------|----------------------------------|
| **REST API ingestion** | Pagination, rate limiting, scheduled HTTP pulls, `mergeSchema` for API version drift | GlobalMart has no enrichment API in scope for `fact_sales` — but you'll meet this pattern on real client projects |
| **Neo4j / Cypher basics** | Nodes, edges, Cypher queries, why graph traversal beats recursive SQL for relationship questions | GlobalMart's product/customer relationships live in flat reference tables, not a graph DB — again, a pattern for your toolkit, not this build |

**The takeaway:** knowing *more* ingestion patterns than your current project needs is normal and valuable — you just need to correctly identify, for a given project, which 1–2 pathways are actually in play. For GlobalMart, that's Lakeflow Connect + Autoloader. Nothing else touches Bronze.

---
## Cross-Pathway Comparison

| Dimension | Postgres CDC (Lakeflow Connect) | ADLS Autoloader |
|-----------|----------------------------------|-------------------|
| **Tables** | `orders`, `order_items` | `products`, `customers`, `addresses`, `payments` |
| **Data shape** | Transactional rows, queried on a cursor | Periodic reference-data file drops |
| **Mechanism** | Query/cursor-based capture (cursor: `updated_at`) | `cloudFiles` streaming source |
| **Exactly-once via** | Cursor's last-seen value | Checkpoint (tracks processed files) |
| **Captures deletes?** | **No** — a deleted row simply stops matching the query | N/A — files are full snapshots, not deltas |
| **Trigger mode** | Manually triggered (no schedule set — see Day 2) | `availableNow` (batch) or `processingTime` (continuous) |
| **Schema evolution** | Managed by Postgres DDL + Lakeflow | `cloudFiles.schemaEvolutionMode = addNewColumns` |
| **Bronze write mode** | Upsert — latest row per key only (history tracking Off / SCD1) | Append (every new file's rows are new) |
| **Audit signature** | Managed by Lakeflow Connect (governed UC table, no manual audit columns needed) | `_source_file`, `_ingested_at` |

---

## Bronze Landing Map — All 6 Tables

Every table below lives in **your catalog**, `bronze` schema — Unity Catalog managed tables, no manual path bookkeeping needed once the external location and connection are set up:

```
<your-catalog>.bronze.orders          ← Upserted, latest state — Lakeflow Connect
<your-catalog>.bronze.order_items     ← Upserted, latest state — Lakeflow Connect

<your-catalog>.bronze.products        ← Auto Loader (append)
<your-catalog>.bronze.customers       ← Auto Loader (append)
<your-catalog>.bronze.addresses       ← Auto Loader (append)
<your-catalog>.bronze.payments        ← Auto Loader (append)
```

**Bronze rules:**
1. Never transform — land raw data as close to source shape as possible
2. Autoloader tables always add audit columns: `_ingested_at`, `_source_file`
3. Autoloader Bronze is append-only, never deleted from directly
4. Lakeflow Connect's Bronze tables are the one exception to "append-only": they're upserted (latest state per key), and a source DELETE is invisible to this pathway — the row simply goes stale, since query/cursor capture never sees it
5. All 6 tables are Delta format, regardless of pathway

---
## What Comes Next

```
Bronze (raw, 6 tables)                    Silver (clean, conformed)        Gold (business-ready)
──────────────────────────────────        ────────────────────────         ────────────────────────
<your-catalog>.bronze.orders           →  <your-catalog>.silver.orders        →    fact_sales
<your-catalog>.bronze.order_items      →  <your-catalog>.silver.order_items   →      (grain: one row per
<your-catalog>.bronze.products         →  <your-catalog>.silver.products           order line item)
<your-catalog>.bronze.customers        →  <your-catalog>.silver.customers     →    dim_customer, dim_product,
<your-catalog>.bronze.addresses        →  <your-catalog>.silver.address            dim_date, dim_address,
<your-catalog>.bronze.payments         →  <your-catalog>.silver.payments           dim_payment_method
```

**Silver transformations (later sessions):**
- Cast data types explicitly (string dates → timestamp)
- Deduplicate ADLS file drops (a re-uploaded file shouldn't create duplicate customers)
- Enrich and standardise column names
- Apply data quality rules before anything reaches Gold

This session's job was to make sure the *shape* of Bronze is crystal clear before we formalize its design rules in the next ILT and build it hands-on right after.

---
## Key Takeaways

1. **GlobalMart has two real ingestion pathways, not four** — Postgres CDC and ADLS Autoloader
2. **Lakeflow Connect > watermark JDBC** for transactional sources — lower source load and far less code to maintain, since Databricks manages the connection, scheduling, and schema handling for you. It does **not** capture hard deletes though — GlobalMart's real pipeline is configured query/cursor-based, the same blind spot as a hand-rolled watermark
3. **Auto Loader > `spark.read` loop** for files — exactly-once via checkpoint, built-in schema evolution
4. **APIs and Neo4j are real patterns you now know** — just not ones this particular pipeline uses; Day 3 ILT 1 was deliberately a side-exploration
5. **Autoloader Bronze is append-only with audit columns; Lakeflow Connect Bronze is upserted** (latest state only, no history, no delete capture) — different write patterns for different source shapes
6. **Six tables, one Bronze layer** — `orders`, `order_items` (Lakeflow Connect) + `products`, `customers`, `address`, `payments` (Autoloader) — all feeding toward `fact_sales`

---

## Discussion Questions

1. *Why can't Auto Loader be used for the Postgres `orders` table?*

2. *A new ADLS file lands every 5 minutes. Should ingestion use `availableNow=True` or `processingTime`? Why?*

3. *GlobalMart's real Lakeflow Connect pipeline can't detect a hard DELETE on `orders`. What would have to change about the pipeline's configuration to fix that — and what would it cost?*

4. *`products.csv` gets a new column `discount_eligible` in next week's drop. What happens to the Bronze write? What config makes this safe?*

5. *If GlobalMart later added a REST weather API for delivery-delay analysis, which pathway from today would it resemble more — Lakeflow Connect or Autoloader? Why?*